# YBT Data Processing Pipeline

This notebook processes the YBT dataset using the exact same methodology as the `data_pipeline_recreation.ipynb` notebook, adapted for YBT-specific characteristics.

## YBT-Specific Adaptations
- **No SPQ data**: The YBT dataset does not contain SPQ questionnaire data
- **Target variable**: Autism target identified by selection of 'autism' in diagnosis column
- **Available questionnaires**: EQ-10, SQR-10, AQ-10 (no SPQ-10)

## Pipeline Overview
1. **Initial Data Exploration**: Understand YBT dataset structure and characteristics
2. **Raw data loading and initial processing**
3. **Target variable creation**: YBT-specific autism diagnosis logic
4. **Missing value handling** (with corrected sex imputation)
5. **Questionnaire scoring** (EQ, SQR, AQ - no SPQ)
6. **Feature engineering** (excluding SPQ-related features)
7. **Data standardization and encoding**
8. **Data balancing** (50/50 split)
9. **Final dataset filtering** (exclude autism cases with AQ < 6)
10. **Experimental setups** adapted for YBT

## Expected Results
- Scientifically rigorous processing pipeline
- Publication-ready validation framework
- Comprehensive model performance analysis
- Clinical interpretation of YBT-specific findings


In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Machine learning libraries
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier, AdaBoostClassifier
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, classification_report, roc_auc_score, 
    f1_score, precision_score, recall_score, precision_recall_curve
)
from sklearn.utils import resample

# Advanced ML libraries
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Set random seeds for reproducibility
np.random.seed(42)
import random
random.seed(42)

print("Libraries imported successfully")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")


## 0. INITIAL YBT DATA EXPLORATION

Before implementing the processing pipeline, we need to understand the YBT dataset structure and characteristics.


In [ ]:
print("="*80)
print("STEP 0: YBT DATASET INITIAL EXPLORATION")
print("="*80)

# Load YBT data
ybt_data_path = '/Users/eb2007/Library/CloudStorage/OneDrive-UniversityofCambridge/Documents/PhD/data/YBT.csv'
print(f"Loading YBT data from: {ybt_data_path}")

try:
    df_ybt = pd.read_csv(ybt_data_path)
    print(f"YBT dataset shape: {df_ybt.shape}")
    print(f"YBT columns: {list(df_ybt.columns)}")
    
    # Basic dataset information
    print(f"\nDataset Info:")
    print(f"  Total rows: {df_ybt.shape[0]:,}")
    print(f"  Total columns: {df_ybt.shape[1]}")
    print(f"  Memory usage: {df_ybt.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    
    # Check for questionnaire columns (YBT-specific naming)
    spq_cols = [col for col in df_ybt.columns if col.startswith('spq_')]
    eq_cols = [col for col in df_ybt.columns if col.startswith('eq10_')]
    sqr_cols = [col for col in df_ybt.columns if col.startswith('sq10_')]
    aq_cols = [col for col in df_ybt.columns if col.startswith('aq_')]
    
    print(f"\nQuestionnaire columns found:")
    print(f"  SPQ columns: {len(spq_cols)} - {spq_cols[:5] if spq_cols else 'None'}")
    print(f"  EQ columns: {len(eq_cols)} - {eq_cols[:5] if eq_cols else 'None'}")
    print(f"  SQR columns: {len(sqr_cols)} - {sqr_cols[:5] if sqr_cols else 'None'}")
    print(f"  AQ columns: {len(aq_cols)} - {aq_cols[:5] if aq_cols else 'None'}")
    
    # Look for diagnosis/target columns
    diagnosis_cols = [col for col in df_ybt.columns if 'diagnosis' in col.lower() or 'autism' in col.lower()]
    print(f"\nDiagnosis/autism columns: {diagnosis_cols}")
    
    # Check for demographic columns (YBT-specific naming)
    demo_cols = ['age', 'sex', 'gender', 'hand', 'edu', 'country']
    available_demo = [col for col in demo_cols if col in df_ybt.columns]
    print(f"Available demographic columns: {available_demo}")
    
    # CRITICAL: Show actual row examples to understand data structure
    print(f"\n" + "="*80)
    print("ACTUAL ROW EXAMPLES - UNDERSTANDING DATA STRUCTURE")
    print("="*80)
    
    # Show first few rows of key columns
    key_cols = ['age', 'sex', 'gender', 'diagnosis_yes_no', 'diagnosis']
    print(f"\nFirst 5 rows of key columns:")
    for col in key_cols:
        if col in df_ybt.columns:
            sample_vals = df_ybt[col].head(5).tolist()
            print(f"  {col}: {sample_vals}")
    
    # Show questionnaire data examples
    if eq_cols:
        print(f"\nFirst 5 rows of EQ questionnaire data:")
        for col in eq_cols[:3]:  # Show first 3 EQ columns
            sample_vals = df_ybt[col].head(5).tolist()
            print(f"  {col}: {sample_vals}")
    
    if sqr_cols:
        print(f"\nFirst 5 rows of SQR questionnaire data:")
        for col in sqr_cols[:3]:  # Show first 3 SQR columns
            sample_vals = df_ybt[col].head(5).tolist()
            print(f"  {col}: {sample_vals}")
    
    if aq_cols:
        print(f"\nFirst 5 rows of AQ questionnaire data:")
        for col in aq_cols[:3]:  # Show first 3 AQ columns
            sample_vals = df_ybt[col].head(5).tolist()
            print(f"  {col}: {sample_vals}")
    
    # Show diagnosis data examples
    if diagnosis_cols:
        print(f"\nFirst 5 rows of diagnosis data:")
        for col in diagnosis_cols:
            sample_vals = df_ybt[col].head(5).tolist()
            print(f"  {col}: {sample_vals}")
    
    # CRITICAL: Analyze data types and values
    print(f"\n" + "="*80)
    print("DATA TYPE AND VALUE ANALYSIS")
    print("="*80)
    
    # Check data types
    print(f"\nData types:")
    dtypes = df_ybt.dtypes.value_counts()
    print(dtypes)
    
    # Check for text vs numeric data in questionnaire columns
    all_questionnaire_cols = eq_cols + sqr_cols + aq_cols
    if all_questionnaire_cols:
        print(f"\nQuestionnaire data analysis:")
        sample_col = all_questionnaire_cols[0]
        print(f"Sample column: {sample_col}")
        
        # Show unique values
        unique_vals = df_ybt[sample_col].value_counts(dropna=False).head(10)
        print(f"Unique values in {sample_col}: {unique_vals.to_dict()}")
        
        # Check if it's text responses
        sample_values = df_ybt[sample_col].dropna().head(10).tolist()
        are_text = any(isinstance(val, str) and any(word in val.lower() for word in ['agree', 'disagree', 'strongly', 'slightly']) for val in sample_values)
        print(f"Contains text responses (agree/disagree): {are_text}")
        
        # Check if it's already numeric
        are_numeric = all(pd.api.types.is_numeric_dtype(df_ybt[col]) for col in all_questionnaire_cols[:3])
        print(f"All questionnaire columns are numeric: {are_numeric}")
        
        # Check if it's binary (0-1)
        if are_numeric:
            sample_binary = df_ybt[sample_col].dropna().head(10).tolist()
            are_binary = all(val in [0, 1] for val in sample_binary if pd.notna(val))
            print(f"Is binary (0-1): {are_binary}")
            
            if are_binary:
                print("✅ Data is already binary - no scoring needed")
            else:
                print("✅ Data is numeric but not binary - scoring needed")
        else:
            print("✅ Data contains text - conversion needed")
    
    # Check missing data patterns
    print(f"\nMissing data summary:")
    missing_data = df_ybt.isnull().sum().sort_values(ascending=False)
    print(f"Columns with missing data: {len(missing_data[missing_data > 0])}")
    if len(missing_data[missing_data > 0]) > 0:
        print("Top 10 columns with most missing data:")
        print(missing_data.head(10))
    
    # Check for metadata contamination
    print(f"\nMetadata contamination check:")
    metadata_cols = []
    for col in df_ybt.columns:
        if df_ybt[col].astype(str).str.contains('ImportId|question', case=False, na=False).any():
            metadata_cols.append(col)
    
    if metadata_cols:
        print(f"Columns with metadata contamination: {len(metadata_cols)}")
        print(f"Sample contaminated columns: {metadata_cols[:5]}")
        
        # Show example metadata
        for col in metadata_cols[:2]:
            sample_metadata = df_ybt[col].astype(str).str.contains('ImportId|question', case=False, na=False)
            metadata_examples = df_ybt[sample_metadata][col].head(3).tolist()
            print(f"  {col} metadata examples: {metadata_examples}")
    else:
        print("No metadata contamination detected")
    
    print(f"\n" + "="*80)
    print("EXPLORATION COMPLETE - READY FOR PROCESSING")
    print("="*80)
    
except FileNotFoundError:
    print(f"ERROR: YBT data file not found at {ybt_data_path}")
    print("Please check the file path and ensure the YBT.csv file exists.")
    print("Creating dummy dataset for demonstration...")
    
    # Create dummy dataset for demonstration
    np.random.seed(42)
    n_samples = 1000
    
    df_ybt = pd.DataFrame({
        'age': np.random.randint(18, 80, n_samples),
        'sex': np.random.choice([1, 2], n_samples),
        'eq_1': np.random.randint(1, 5, n_samples),
        'eq_2': np.random.randint(1, 5, n_samples),
        'eq_3': np.random.randint(1, 5, n_samples),
        'eq_4': np.random.randint(1, 5, n_samples),
        'eq_5': np.random.randint(1, 5, n_samples),
        'eq_6': np.random.randint(1, 5, n_samples),
        'eq_7': np.random.randint(1, 5, n_samples),
        'eq_8': np.random.randint(1, 5, n_samples),
        'eq_9': np.random.randint(1, 5, n_samples),
        'eq_10': np.random.randint(1, 5, n_samples),
        'sqr_1': np.random.randint(1, 5, n_samples),
        'sqr_2': np.random.randint(1, 5, n_samples),
        'sqr_3': np.random.randint(1, 5, n_samples),
        'sqr_4': np.random.randint(1, 5, n_samples),
        'sqr_5': np.random.randint(1, 5, n_samples),
        'sqr_6': np.random.randint(1, 5, n_samples),
        'sqr_7': np.random.randint(1, 5, n_samples),
        'sqr_8': np.random.randint(1, 5, n_samples),
        'sqr_9': np.random.randint(1, 5, n_samples),
        'sqr_10': np.random.randint(1, 5, n_samples),
        'aq_1': np.random.randint(1, 5, n_samples),
        'aq_2': np.random.randint(1, 5, n_samples),
        'aq_3': np.random.randint(1, 5, n_samples),
        'aq_4': np.random.randint(1, 5, n_samples),
        'aq_5': np.random.randint(1, 5, n_samples),
        'aq_6': np.random.randint(1, 5, n_samples),
        'aq_7': np.random.randint(1, 5, n_samples),
        'aq_8': np.random.randint(1, 5, n_samples),
        'aq_9': np.random.randint(1, 5, n_samples),
        'aq_10': np.random.randint(1, 5, n_samples),
    })
    
    # Add diagnosis column with autism selection
    diagnosis_options = ['autism', 'adhd', 'anxiety', 'depression', 'none']
    df_ybt['diagnosis_selection'] = df_ybt.apply(
        lambda x: 'autism' if np.random.random() < 0.1 else np.random.choice(diagnosis_options), 
        axis=1
    )
    
    print(f"Dummy YBT dataset created with shape: {df_ybt.shape}")
    print(f"Dummy columns: {list(df_ybt.columns)}")

## 1. YBT DATA PROCESSING PIPELINE

### A. Raw Data Loading and Initial Processing


In [ ]:
print("="*80)
print("STEP A: YBT RAW DATA LOADING AND INITIAL PROCESSING")
print("="*80)

# Use the YBT dataset from exploration
df = df_ybt.copy()
print(f"Starting with YBT dataset shape: {df.shape}")

# CRITICAL: Drop metadata rows (first 2 rows) from ALL columns
print("\nDropping metadata rows (first 2 rows) from all columns...")
df = df.iloc[2:].reset_index(drop=True)
print(f"After dropping metadata rows: {df.shape}")

# Check what the clean data looks like now
print(f"\nSample of clean data:")
sample_cols = ['age', 'sex', 'diagnosis_yes_no', 'diagnosis', 'eq10_1', 'aq_1']
for col in sample_cols:
    if col in df.columns:
        sample_vals = df[col].dropna().head(3).tolist()
        print(f"  {col}: {sample_vals}")

# Remove columns with mostly missing data
print(f"\nAnalyzing missing data patterns...")
missing_data = df.isnull().sum().sort_values(ascending=False)
total_rows = len(df)

print(f"Missing data analysis:")
print(f"  Total rows: {total_rows}")
print(f"  Columns with missing data: {len(missing_data[missing_data > 0])}")

# Define threshold for dropping columns (e.g., >80% missing)
missing_threshold = 0.80  # Drop columns with >80% missing data
print(f"  Missing data threshold: {missing_threshold*100}%")

# Identify columns to drop
columns_to_drop = []
for col in df.columns:
    missing_count = missing_data[col]
    missing_percentage = missing_count / total_rows
    
    if missing_percentage > missing_threshold:
        columns_to_drop.append(col)
        print(f"    DROP: {col} - {missing_count}/{total_rows} ({missing_percentage*100:.1f}% missing)")

print(f"\nDropping {len(columns_to_drop)} columns with >{missing_threshold*100}% missing data...")
df = df.drop(columns=columns_to_drop)
print(f"After dropping high missing columns: {df.shape}")

# Check remaining missing data
print(f"\nRemaining missing data summary:")
remaining_missing = df.isnull().sum().sort_values(ascending=False)
print(f"  Columns with missing data: {len(remaining_missing[remaining_missing > 0])}")
if len(remaining_missing[remaining_missing > 0]) > 0:
    print("  Top 10 columns with most missing data:")
    print(remaining_missing.head(10))

# Check data types after cleaning
print(f"\nData types after cleaning:")
dtypes = df.dtypes.value_counts()
print(dtypes)

# Check questionnaire columns availability
eq_cols = [col for col in df.columns if col.startswith('eq10_')]
sqr_cols = [col for col in df.columns if col.startswith('sq10_')]
aq_cols = [col for col in df.columns if col.startswith('aq_')]

print(f"\nQuestionnaire columns after cleaning:")
print(f"  EQ columns: {len(eq_cols)} - {eq_cols}")
print(f"  SQR columns: {len(sqr_cols)} - {sqr_cols}")
print(f"  AQ columns: {len(aq_cols)} - {aq_cols}")

# Check diagnosis columns
diagnosis_cols = [col for col in df.columns if 'diagnosis' in col.lower()]
print(f"  Diagnosis columns: {diagnosis_cols}")

# Show sample values from clean questionnaire data
if eq_cols:
    print(f"\nSample clean questionnaire values:")
    sample_col = eq_cols[0]
    sample_vals = df[sample_col].dropna().head(5).tolist()
    print(f"  {sample_col}: {sample_vals}")

# Show sample values from clean diagnosis data
if diagnosis_cols:
    print(f"\nSample clean diagnosis values:")
    for col in diagnosis_cols:
        sample_vals = df[col].dropna().head(5).tolist()
        print(f"  {col}: {sample_vals}")

# ADDED: Print all column names
print(f"\n" + "="*80)
print("ALL COLUMN NAMES AFTER CLEANING")
print("="*80)
print(f"Total columns: {len(df.columns)}")
print(f"Column names:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col}")

print(f"\nStep A complete. Dataset shape: {df.shape}")
print("✅ Metadata rows removed")
print("✅ High missing data columns removed")
print("✅ Data ready for target variable creation")

### B. Missing Value Handling


In [ ]:
print("="*80)
print("STEP B: YBT MISSING VALUE HANDLING")
print("="*80)

# Clean age column first
if 'age' in df.columns:
    print("Cleaning age column...")
    # Convert to numeric
    df['age'] = pd.to_numeric(df['age'], errors='coerce')
    print(f"Age column cleaned. Sample values: {df['age'].dropna().head(5).tolist()}")

# Clean demographic columns
demographic_cols = ['sex', 'gender', 'hand', 'edu', 'country']
available_demo_cols = [col for col in demographic_cols if col in df.columns]
print(f"\nCleaning demographic columns: {available_demo_cols}")

for col in available_demo_cols:
    if col in df.columns:
        # Fill missing values with 'unknown'
        df[col] = df[col].fillna('unknown')
        print(f"  {col}: {df[col].isnull().sum()} missing values remaining")

# Focus on questionnaire missing data analysis
questionnaire_cols = [col for col in df.columns if any(q in col for q in ['eq10_', 'sq10_', 'aq_'])]
print(f"\nQUESTIONNAIRE MISSING DATA ANALYSIS: {len(questionnaire_cols)} columns")

if questionnaire_cols:
    # Analyze missing data patterns in questionnaire responses
    print(f"\nMissing data analysis by questionnaire type:")
    
    # Calculate missing data for each questionnaire type
    for q_type in ['eq10_', 'sq10_', 'aq_']:
        q_cols = [col for col in questionnaire_cols if col.startswith(q_type)]
        if q_cols:
            # Calculate missing data for this questionnaire type
            q_missing = df[q_cols].isnull().sum(axis=1)
            q_total = len(q_cols)
            q_missing_percentage = (q_missing / q_total) * 100
            
            print(f"\n{q_type} questionnaire analysis:")
            print(f"  Total questions: {q_total}")
            print(f"  Mean missing per person: {q_missing_percentage.mean():.1f}%")
            print(f"  Median missing per person: {q_missing_percentage.median():.1f}%")
            print(f"  Max missing per person: {q_missing_percentage.max():.1f}%")
            
            # Show distribution of missing data
            missing_distribution = q_missing_percentage.value_counts().sort_index()
            print(f"  Missing data distribution:")
            print(f"    0% missing: {missing_distribution.get(0.0, 0)} people")
            print(f"    1-25% missing: {q_missing_percentage[(q_missing_percentage > 0) & (q_missing_percentage <= 25)].count()} people")
            print(f"    26-50% missing: {q_missing_percentage[(q_missing_percentage > 25) & (q_missing_percentage <= 50)].count()} people")
            print(f"    51-75% missing: {q_missing_percentage[(q_missing_percentage > 50) & (q_missing_percentage <= 75)].count()} people")
            print(f"    76-100% missing: {q_missing_percentage[q_missing_percentage > 75].count()} people")
    
    # Calculate overall missing data per person across ALL questionnaires
    print(f"\nOVERALL QUESTIONNAIRE MISSING DATA ANALYSIS:")
    all_q_missing = df[questionnaire_cols].isnull().sum(axis=1)
    all_q_total = len(questionnaire_cols)
    all_q_missing_percentage = (all_q_missing / all_q_total) * 100
    
    print(f"  Total questionnaire questions: {all_q_total}")
    print(f"  Mean missing per person: {all_q_missing_percentage.mean():.1f}%")
    print(f"  Median missing per person: {all_q_missing_percentage.median():.1f}%")
    print(f"  Max missing per person: {all_q_missing_percentage.max():.1f}%")
    
    # Show overall distribution
    overall_distribution = all_q_missing_percentage.value_counts().sort_index()
    print(f"  Overall missing data distribution:")
    print(f"    0% missing: {overall_distribution.get(0.0, 0)} people")
    print(f"    1-25% missing: {all_q_missing_percentage[(all_q_missing_percentage > 0) & (all_q_missing_percentage <= 25)].count()} people")
    print(f"    26-50% missing: {all_q_missing_percentage[(all_q_missing_percentage > 25) & (all_q_missing_percentage <= 50)].count()} people")
    print(f"    51-75% missing: {all_q_missing_percentage[(all_q_missing_percentage > 50) & (all_q_missing_percentage <= 75)].count()} people")
    print(f"    76-100% missing: {all_q_missing_percentage[all_q_missing_percentage > 75].count()} people")
    
    # DECISION: Drop individuals with high missing questionnaire data
    print(f"\nDECISION: Dropping individuals with high missing questionnaire data...")
    
    # Define threshold for dropping individuals
    # Best practice: Drop individuals with >50% missing questionnaire data
    # This ensures we keep people who answered most questions
    high_missing_threshold = 50  # Drop individuals with >50% missing questionnaire data
    
    print(f"  Threshold: Drop individuals with >{high_missing_threshold}% missing questionnaire data")
    print(f"  Rationale: Keep people who answered most questions for reliable analysis")
    
    # Identify individuals to drop
    high_missing_individuals = all_q_missing_percentage > high_missing_threshold
    individuals_to_drop = high_missing_individuals.sum()
    
    print(f"  Individuals with >{high_missing_threshold}% missing data: {individuals_to_drop}")
    print(f"  Individuals with <{high_missing_threshold}% missing data: {(all_q_missing_percentage <= high_missing_threshold).sum()}")
    
    if individuals_to_drop > 0:
        print(f"  Dropping {individuals_to_drop} individuals with high missing data...")
        df_before_drop = df.copy()
        df = df[~high_missing_individuals]
        rows_dropped = len(df_before_drop) - len(df)
        print(f"  Rows dropped: {rows_dropped}")
        print(f"  After dropping high missing data individuals: {df.shape}")
        
        # Check if we still have enough data
        if len(df) < 1000:
            print("  WARNING: Very few rows remaining after dropping high missing data individuals")
            print("  Consider using a lower threshold or imputation instead")
        else:
            print(f"  Sufficient data remaining: {len(df)} rows")
    else:
        print(f"  No individuals with >{high_missing_threshold}% missing data found")
        print(f"  Keeping all {len(df)} individuals")
        
else:
    print("No questionnaire columns found")

# Check remaining missing values
print(f"\nRemaining missing values after processing:")
remaining_missing = df.isnull().sum().sort_values(ascending=False)
print(f"  Columns with missing data: {len(remaining_missing[remaining_missing > 0])}")
if len(remaining_missing[remaining_missing > 0]) > 0:
    print("  Top 10 columns with most missing data:")
    print(remaining_missing.head(10))

print(f"\nStep B complete. Dataset shape: {df.shape}")
print("✅ High missing data individuals removed")
print("✅ Data ready for questionnaire scoring")

### C. Questionnaire Scoring and Totals (YBT Adapted - No SPQ)


In [ ]:
print("="*80)
print("STEP C: YBT QUESTIONNAIRE SCORING")
print("="*80)

# Define response mapping for all questionnaires
print("Setting up response mapping...")

# Response mapping: text responses to numeric values
response_mapping = {
    'strongly agree': 4,
    'slightly agree': 3, 
    'slightly disagree': 2,
    'strongly disagree': 1
}

print(f"Response mapping: {response_mapping}")

# Get questionnaire columns
eq_cols = [col for col in df.columns if col.startswith('eq10_')]
sqr_cols = [col for col in df.columns if col.startswith('sq10_')]
aq_cols = [col for col in df.columns if col.startswith('aq_')]

print(f"\nQuestionnaire columns found:")
print(f"  EQ columns: {len(eq_cols)} - {eq_cols}")
print(f"  SQR columns: {len(sqr_cols)} - {sqr_cols}")
print(f"  AQ columns: {len(aq_cols)} - {aq_cols}")

# STEP 1: Convert text responses to numeric for all questionnaires
print(f"\nSTEP 1: Converting text responses to numeric...")

all_questionnaire_cols = eq_cols + sqr_cols + aq_cols
for col in all_questionnaire_cols:
    if col in df.columns:
        # Convert text responses to numeric using mapping
        df[col] = df[col].map(response_mapping)
        print(f"  Converted {col}: {df[col].dropna().head(3).tolist()}")

# STEP 2: EQ-10 Scoring (Binary 0-1 with reverse-scoring)
print(f"\nSTEP 2: EQ-10 Scoring (Binary 0-1)...")

if eq_cols:
    # EQ-10 scoring: Binary 0-1 per item
    # Items 1,2,4,5,6,7,8,9,10: Agree responses (3,4) = 1 point
    # Items 3: Disagree responses (1,2) = 1 point (reverse-scored)
    
    eq_reverse_items = [3]  # Items that need reverse scoring
    
    for i, col in enumerate(eq_cols, 1):
        if col in df.columns:
            if i in eq_reverse_items:
                # Reverse scoring: disagree = 1, agree = 0
                df[col] = df[col].apply(lambda x: 1 if x in [1, 2] else 0 if x in [3, 4] else np.nan)
                print(f"  {col} (reverse): {df[col].dropna().head(3).tolist()}")
            else:
                # Normal scoring: agree = 1, disagree = 0
                df[col] = df[col].apply(lambda x: 1 if x in [3, 4] else 0 if x in [1, 2] else np.nan)
                print(f"  {col} (normal): {df[col].dropna().head(3).tolist()}")
    
    # Calculate EQ total
    df['eq_total'] = df[eq_cols].sum(axis=1)
    print(f"  EQ total range: {df['eq_total'].min()} to {df['eq_total'].max()}")
    print(f"  EQ total mean: {df['eq_total'].mean():.2f}")

# STEP 3: SQR-10 Scoring (Binary 0-1 with reverse-scoring)
print(f"\nSTEP 3: SQR-10 Scoring (Binary 0-1)...")

if sqr_cols:
    # SQR-10 scoring: Binary 0-1 per item
    # Items 1,3,5,7,9: Agree responses (3,4) = 1 point
    # Items 2,4,6,8,10: Disagree responses (1,2) = 1 point (reverse-scored)
    
    sqr_reverse_items = [2, 4, 6, 8, 10]  # Items that need reverse scoring
    
    for i, col in enumerate(sqr_cols, 1):
        if col in df.columns:
            if i in sqr_reverse_items:
                # Reverse scoring: disagree = 1, agree = 0
                df[col] = df[col].apply(lambda x: 1 if x in [1, 2] else 0 if x in [3, 4] else np.nan)
                print(f"  {col} (reverse): {df[col].dropna().head(3).tolist()}")
            else:
                # Normal scoring: agree = 1, disagree = 0
                df[col] = df[col].apply(lambda x: 1 if x in [3, 4] else 0 if x in [1, 2] else np.nan)
                print(f"  {col} (normal): {df[col].dropna().head(3).tolist()}")
    
    # Calculate SQR total
    df['sqr_total'] = df[sqr_cols].sum(axis=1)
    print(f"  SQR total range: {df['sqr_total'].min()} to {df['sqr_total'].max()}")
    print(f"  SQR total mean: {df['sqr_total'].mean():.2f}")

# STEP 4: AQ-10 Scoring (Binary 0-1 with ARC reverse-scoring)
print(f"\nSTEP 4: AQ-10 Scoring (Binary 0-1 with ARC correction)...")

if aq_cols:
    # AQ-10 scoring: Binary 0-1 per item with ARC correction
    # Items 1,7,8,10: Agree responses (3,4) = 1 point (autistic traits)
    # Items 2,3,4,5,6,9: Disagree responses (1,2) = 1 point (autistic traits, reverse-scored)
    
    aq_reverse_items = [2, 3, 4, 5, 6, 9]  # Items that need reverse scoring
    
    for i, col in enumerate(aq_cols, 1):
        if col in df.columns:
            if i in aq_reverse_items:
                # Reverse scoring: disagree = 1, agree = 0
                df[col] = df[col].apply(lambda x: 1 if x in [1, 2] else 0 if x in [3, 4] else np.nan)
                print(f"  {col} (reverse): {df[col].dropna().head(3).tolist()}")
            else:
                # Normal scoring: agree = 1, disagree = 0
                df[col] = df[col].apply(lambda x: 1 if x in [3, 4] else 0 if x in [1, 2] else np.nan)
                print(f"  {col} (normal): {df[col].dropna().head(3).tolist()}")
    
    # Calculate AQ total
    df['aq_total'] = df[aq_cols].sum(axis=1)
    print(f"  AQ total range: {df['aq_total'].min()} to {df['aq_total'].max()}")
    print(f"  AQ total mean: {df['aq_total'].mean():.2f}")

# STEP 5: Validation and Summary
print(f"\nSTEP 5: Validation and Summary...")

# Check for any remaining non-numeric values
print(f"\nChecking for data quality issues:")
for col in all_questionnaire_cols:
    if col in df.columns:
        non_numeric = df[col].apply(lambda x: not pd.api.types.is_numeric_dtype(type(x)) if pd.notna(x) else False).sum()
        if non_numeric > 0:
            print(f"  WARNING: {col} has {non_numeric} non-numeric values")

# Show questionnaire totals summary
print(f"\nQuestionnaire totals summary:")
if 'eq_total' in df.columns:
    print(f"  EQ total: {df['eq_total'].min()}-{df['eq_total'].max()} (mean: {df['eq_total'].mean():.2f})")
if 'sqr_total' in df.columns:
    print(f"  SQR total: {df['sqr_total'].min()}-{df['sqr_total'].max()} (mean: {df['sqr_total'].mean():.2f})")
if 'aq_total' in df.columns:
    print(f"  AQ total: {df['aq_total'].min()}-{df['aq_total'].max()} (mean: {df['aq_total'].mean():.2f})")

# Check missing data in totals
print(f"\nMissing data in totals:")
if 'eq_total' in df.columns:
    print(f"  EQ total missing: {df['eq_total'].isnull().sum()}")
if 'sqr_total' in df.columns:
    print(f"  SQR total missing: {df['sqr_total'].isnull().sum()}")
if 'aq_total' in df.columns:
    print(f"  AQ total missing: {df['aq_total'].isnull().sum()}")

print(f"\nStep C complete. Dataset shape: {df.shape}")
print("✅ All questionnaires scored (binary 0-1)")
print("✅ Reverse-scoring applied correctly")
print("✅ Total scores calculated")
print("✅ Data ready for target variable creation")

### D. Target variable creation

In [ ]:
print("="*80)
print("STEP D: YBT TARGET VARIABLE CREATION")
print("="*80)

# Check diagnosis columns
diagnosis_cols = [col for col in df.columns if 'diagnosis' in col.lower()]
print(f"Diagnosis columns found: {diagnosis_cols}")

# Show sample diagnosis data
print(f"\nSample diagnosis data:")
for col in diagnosis_cols:
    if col in df.columns:
        sample_vals = df[col].dropna().head(10).tolist()
        print(f"  {col}: {sample_vals}")

# STEP 1: Clean diagnosis_yes_no column
print(f"\nSTEP 1: Cleaning diagnosis_yes_no column...")

if 'diagnosis_yes_no' in df.columns:
    # Show unique values before cleaning
    unique_before = df['diagnosis_yes_no'].value_counts(dropna=False)
    print(f"  Unique values before cleaning: {unique_before.to_dict()}")
    
    # Convert Yes/No to 1/0
    df['diagnosis_yes_no'] = df['diagnosis_yes_no'].map({'Yes': 1, 'No': 0})
    
    # Show unique values after cleaning
    unique_after = df['diagnosis_yes_no'].value_counts(dropna=False)
    print(f"  Unique values after cleaning: {unique_after.to_dict()}")
    
    # Check for any remaining non-numeric values
    non_numeric = df['diagnosis_yes_no'].apply(lambda x: not pd.api.types.is_numeric_dtype(type(x)) if pd.notna(x) else False).sum()
    if non_numeric > 0:
        print(f"  WARNING: {non_numeric} non-numeric values remaining")
    else:
        print(f"  ✅ All values converted to numeric")

# STEP 2: Create autism target variable
print(f"\nSTEP 2: Creating autism target variable...")

# YBT-specific logic: Target is 1 if 'autism' appears in ANY diagnosis column
if 'diagnosis' in df.columns:
    print(f"  Using diagnosis column for autism detection...")
    
    # Check for autism in diagnosis column
    autism_in_diagnosis = df['diagnosis'].str.contains('autism', case=False, na=False)
    
    # Also check diagnosis_69_TEXT if it exists
    autism_in_diagnosis_text = False
    if 'diagnosis_69_TEXT' in df.columns:
        autism_in_diagnosis_text = df['diagnosis_69_TEXT'].str.contains('autism', case=False, na=False)
    
    # Create target: 1 if autism appears in ANY diagnosis column
    df['autism_target'] = (autism_in_diagnosis | autism_in_diagnosis_text).astype(int)
    
    print(f"  ✅ Autism target created using ANY diagnosis column containing 'autism'")
    
    # Show sample autism cases
    autism_cases = df[df['autism_target'] == 1]
    if len(autism_cases) > 0:
        print(f"  Sample autism cases:")
        sample_autism_diagnoses = autism_cases['diagnosis'].dropna().head(5).tolist()
        print(f"    {sample_autism_diagnoses}")
    
    # Show sample non-autism cases
    non_autism_cases = df[df['autism_target'] == 0]
    if len(non_autism_cases) > 0:
        print(f"  Sample non-autism cases:")
        sample_non_autism_diagnoses = non_autism_cases['diagnosis'].dropna().head(5).tolist()
        print(f"    {sample_non_autism_diagnoses}")
    
else:
    print(f"  ERROR: No diagnosis column found!")
    print(f"  Available columns: {list(df.columns)}")
    # Create dummy target for demonstration
    df['autism_target'] = np.random.choice([0, 1], len(df), p=[0.9, 0.1])
    print(f"  Created dummy autism_target for demonstration")

# STEP 3: Validate target variable creation
print(f"\nSTEP 3: Target variable validation...")

# Show target distribution
target_distribution = df['autism_target'].value_counts()
print(f"  Target distribution:")
print(f"    Autism cases (1): {target_distribution.get(1, 0)}")
print(f"    Non-autism cases (0): {target_distribution.get(0, 0)}")

# Calculate prevalence
total_cases = len(df)
autism_cases = target_distribution.get(1, 0)
autism_prevalence = autism_cases / total_cases

print(f"  Autism prevalence: {autism_prevalence*100:.2f}%")

# Check if prevalence is reasonable
if autism_prevalence < 0.01:
    print(f"  ⚠️  VERY LOW PREVALENCE: {autism_prevalence*100:.2f}%")
    print(f"  This may indicate issues with target variable creation")
    print(f"  Expected prevalence: 1-5% in general population")
elif autism_prevalence > 0.1:
    print(f"  ⚠️  HIGH PREVALENCE: {autism_prevalence*100:.2f}%")
    print(f"  This may indicate sampling bias or target creation issues")
    print(f"  Expected prevalence: 1-5% in general population")
else:
    print(f"  ✅ REASONABLE PREVALENCE: {autism_prevalence*100:.2f}%")
    print(f"  This is within expected range for autism prevalence")

# STEP 4: Cross-validation with diagnosis_yes_no
print(f"\nSTEP 4: Cross-validation with diagnosis_yes_no...")

if 'diagnosis_yes_no' in df.columns:
    # Check if autism cases have diagnosis_yes_no = 1
    autism_cases = df[df['autism_target'] == 1]
    if len(autism_cases) > 0:
        autism_yes_no = autism_cases['diagnosis_yes_no'].value_counts()
        print(f"  Autism cases diagnosis_yes_no distribution:")
        print(f"    {autism_yes_no.to_dict()}")
        
        # Validate that autism cases have diagnosis_yes_no = 1
        autism_with_yes = len(autism_cases[autism_cases['diagnosis_yes_no'] == 1])
        autism_with_no = len(autism_cases[autism_cases['diagnosis_yes_no'] == 0])
        
        print(f"  Autism cases with diagnosis_yes_no=1: {autism_with_yes}")
        print(f"  Autism cases with diagnosis_yes_no=0: {autism_with_no}")
        
        if autism_with_no > 0:
            print(f"  ⚠️  WARNING: Some autism cases have diagnosis_yes_no=0")
            print(f"  This may indicate issues with target variable creation")
        else:
            print(f"  ✅ All autism cases have diagnosis_yes_no=1")
    
    # Check if non-autism cases have diagnosis_yes_no = 0
    non_autism_cases = df[df['autism_target'] == 0]
    if len(non_autism_cases) > 0:
        non_autism_yes_no = non_autism_cases['diagnosis_yes_no'].value_counts()
        print(f"  Non-autism cases diagnosis_yes_no distribution:")
        print(f"    {non_autism_yes_no.to_dict()}")
        
        # Check if any non-autism cases have autism in diagnosis
        if 'diagnosis' in df.columns:
            non_autism_with_autism = non_autism_cases[non_autism_cases['diagnosis'].str.contains('autism', case=False, na=False)]
            if len(non_autism_with_autism) > 0:
                print(f"  ⚠️  WARNING: {len(non_autism_with_autism)} non-autism cases contain 'autism' in diagnosis")
                print(f"  This may indicate issues with target variable creation")
            else:
                print(f"  ✅ No non-autism cases contain 'autism' in diagnosis")

# STEP 5: Summary and final validation
print(f"\nSTEP 5: Final validation summary...")

print(f"  Dataset shape: {df.shape}")
print(f"  Target variable: autism_target")
print(f"  Target distribution: {target_distribution.to_dict()}")
print(f"  Autism prevalence: {autism_prevalence*100:.2f}%")

# Check for missing target values
missing_target = df['autism_target'].isnull().sum()
if missing_target > 0:
    print(f"  ⚠️  WARNING: {missing_target} missing target values")
else:
    print(f"  ✅ No missing target values")

# Show sample of final data
print(f"\nSample of final data:")
sample_cols = ['age', 'sex', 'diagnosis_yes_no', 'diagnosis', 'autism_target', 'eq_total', 'aq_total']
for col in sample_cols:
    if col in df.columns:
        sample_vals = df[col].dropna().head(3).tolist()
        print(f"  {col}: {sample_vals}")

print(f"\nStep D complete. Dataset shape: {df.shape}")
print("✅ Target variable created successfully")
print("✅ Target validation completed")
print("✅ Data ready for feature engineering")

In [ ]:
# AQ Score Analysis by Autism Status (After Step D)
print("="*80)
print("AQ SCORE ANALYSIS BY AUTISM STATUS")
print("="*80)

if 'aq_total' in df.columns and 'autism_target' in df.columns:
    # Calculate AQ means by autism status
    autism_cases = df[df['autism_target'] == 1]
    non_autism_cases = df[df['autism_target'] == 0]
    
    if len(autism_cases) > 0 and len(non_autism_cases) > 0:
        autism_aq_mean = autism_cases['aq_total'].mean()
        non_autism_aq_mean = non_autism_cases['aq_total'].mean()
        
        print(f"AQ Score Analysis:")
        print(f"  Autism cases (n={len(autism_cases)}): Mean AQ = {autism_aq_mean:.2f}")
        print(f"  Non-autism cases (n={len(non_autism_cases)}): Mean AQ = {non_autism_aq_mean:.2f}")
        print(f"  Difference: {autism_aq_mean - non_autism_aq_mean:.2f}")
        
        # Clinical interpretation
        if autism_aq_mean > non_autism_aq_mean:
            print(f"  ✅ CLINICALLY CORRECT: Autism cases have HIGHER AQ scores")
        else:
            print(f"  ❌ CLINICALLY COUNTERINTUITIVE: Autism cases have LOWER AQ scores")
            print(f"  ⚠️  This suggests potential issues with:")
            print(f"     - AQ scoring logic")
            print(f"     - Target variable creation")
            print(f"     - Data processing")
        
        # Show AQ distribution
        print(f"\nAQ Score Distribution:")
        print(f"  Autism cases AQ range: {autism_cases['aq_total'].min():.1f} - {autism_cases['aq_total'].max():.1f}")
        print(f"  Non-autism cases AQ range: {non_autism_cases['aq_total'].min():.1f} - {non_autism_cases['aq_total'].max():.1f}")
        
        # Show sample AQ scores
        print(f"\nSample AQ scores:")
        autism_aq_sample = autism_cases['aq_total'].head(5).tolist()
        non_autism_aq_sample = non_autism_cases['aq_total'].head(5).tolist()
        print(f"  Autism cases: {autism_aq_sample}")
        print(f"  Non-autism cases: {non_autism_aq_sample}")
        
    else:
        print(f"  ⚠️  Cannot analyze AQ scores: Missing autism or non-autism cases")
        print(f"  Autism cases: {len(autism_cases)}")
        print(f"  Non-autism cases: {len(non_autism_cases)}")
else:
    print(f"  ⚠️  Cannot analyze AQ scores:")
    print(f"  aq_total column present: {'aq_total' in df.columns}")
    print(f"  autism_target column present: {'autism_target' in df.columns}")

print("\n" + "="*80)

### E. Feature Engineering (YBT Adapted - No SPQ Features)


In [ ]:
print("="*80)
print("STEP E: YBT FEATURE ENGINEERING")
print("="*80)

# Check current dataset shape
print(f"Starting dataset shape: {df.shape}")

# STEP 1: Age-based features
print(f"\nSTEP 1: Age-based feature engineering...")

if 'age' in df.columns:
    # Age groups
    df['age_group'] = pd.cut(df['age'], bins=[0, 25, 35, 45, 55, 100], 
                             labels=['18-25', '26-35', '36-45', '46-55', '56+'])
    
    # Age squared (non-linear relationship)
    df['age_squared'] = df['age'] ** 2
    
    # Log age (for skewed distributions)
    df['age_log'] = np.log(df['age'])
    
    print(f"  ✅ Age groups created: {df['age_group'].value_counts().to_dict()}")
    print(f"  ✅ Age squared range: {df['age_squared'].min():.1f} to {df['age_squared'].max():.1f}")
    print(f"  ✅ Log age range: {df['age_log'].min():.2f} to {df['age_log'].max():.2f}")

# STEP 2: Demographic feature engineering
print(f"\nSTEP 2: Demographic feature engineering...")

# Sex encoding
if 'sex' in df.columns:
    # Convert to numeric codes
    df['sex_num'] = pd.Categorical(df['sex']).codes
    print(f"  ✅ Sex encoded: {df['sex'].value_counts().to_dict()}")
    print(f"  ✅ Sex numeric codes: {df['sex_num'].value_counts().to_dict()}")

# Gender encoding
if 'gender' in df.columns:
    df['gender_num'] = pd.Categorical(df['gender']).codes
    print(f"  ✅ Gender encoded: {df['gender'].value_counts().to_dict()}")

# Handedness encoding
if 'hand' in df.columns:
    df['hand_num'] = pd.Categorical(df['hand']).codes
    print(f"  ✅ Handedness encoded: {df['hand'].value_counts().to_dict()}")

# Education encoding
if 'edu' in df.columns:
    df['edu_num'] = pd.Categorical(df['edu']).codes
    print(f"  ✅ Education encoded: {df['edu'].value_counts().to_dict()}")

# Country encoding
if 'country' in df.columns:
    df['country_num'] = pd.Categorical(df['country']).codes
    print(f"  ✅ Country encoded: {df['country'].value_counts().to_dict()}")

# STEP 3: Questionnaire interaction features (NO AQ interactions)
print(f"\nSTEP 3: Questionnaire interaction features (EQ-SQR only)...")

# EQ-SQR interaction only (no AQ interactions to prevent data leakage)
if 'eq_total' in df.columns and 'sqr_total' in df.columns:
    df['eq_sqr_interaction'] = df['eq_total'] * df['sqr_total']
    print(f"  ✅ EQ-SQR interaction created: {df['eq_sqr_interaction'].min():.1f} to {df['eq_sqr_interaction'].max():.1f}")

# STEP 4: Questionnaire ratios (NO AQ ratios)
print(f"\nSTEP 4: Questionnaire ratios (EQ-SQR only)...")

# EQ-SQR ratio only (no AQ ratios to prevent data leakage)
if 'eq_total' in df.columns and 'sqr_total' in df.columns:
    df['eq_sqr_ratio'] = df['eq_total'] / (df['sqr_total'] + 1)  # +1 to avoid division by zero
    print(f"  ✅ EQ-SQR ratio created: {df['eq_sqr_ratio'].min():.2f} to {df['eq_sqr_ratio'].max():.2f}")

# STEP 5: Threshold-based features (INCLUDING AQ threshold for exclusion criteria)
print(f"\nSTEP 5: Threshold-based features...")

# High AQ threshold (clinical cutoff) - NEEDED for sample exclusion criteria
if 'aq_total' in df.columns:
    df['high_aq'] = (df['aq_total'] >= 6).astype(int)
    print(f"  ✅ High AQ threshold (≥6): {df['high_aq'].value_counts().to_dict()}")
    print(f"  ⚠️  NOTE: AQ threshold kept for sample exclusion criteria")

# High EQ threshold
if 'eq_total' in df.columns:
    df['high_eq'] = (df['eq_total'] >= 7).astype(int)
    print(f"  ✅ High EQ threshold (≥7): {df['high_eq'].value_counts().to_dict()}")

# High SQR threshold
if 'sqr_total' in df.columns:
    df['high_sqr'] = (df['sqr_total'] >= 6).astype(int)
    print(f"  ✅ High SQR threshold (≥6): {df['high_sqr'].value_counts().to_dict()}")

# STEP 6: Age-questionnaire interactions (NO AQ interactions)
print(f"\nSTEP 6: Age-questionnaire interactions (EQ-SQR only)...")

if 'age' in df.columns:
    # Age-EQ interaction
    if 'eq_total' in df.columns:
        df['age_x_eq'] = df['age'] * df['eq_total']
        print(f"  ✅ Age-EQ interaction created: {df['age_x_eq'].min():.1f} to {df['age_x_eq'].max():.1f}")
    
    # Age-SQR interaction
    if 'sqr_total' in df.columns:
        df['age_x_sqr'] = df['age'] * df['sqr_total']
        print(f"  ✅ Age-SQR interaction created: {df['age_x_sqr'].min():.1f} to {df['age_x_sqr'].max():.1f}")

# STEP 7: Combined questionnaire features (NO AQ)
print(f"\nSTEP 7: Combined questionnaire features (EQ-SQR only)...")

# Total questionnaire score (EQ + SQR only)
questionnaire_totals = ['eq_total', 'sqr_total']
available_totals = [col for col in questionnaire_totals if col in df.columns]
if available_totals:
    df['total_questionnaire_score'] = df[available_totals].sum(axis=1)
    print(f"  ✅ Total questionnaire score (EQ+SQR): {df['total_questionnaire_score'].min():.1f} to {df['total_questionnaire_score'].max():.1f}")

# Questionnaire balance (EQ vs SQR)
if 'eq_total' in df.columns and 'sqr_total' in df.columns:
    df['eq_sqr_balance'] = df['eq_total'] - df['sqr_total']
    print(f"  ✅ EQ-SQR balance: {df['eq_sqr_balance'].min():.1f} to {df['eq_sqr_balance'].max():.1f}")

# STEP 8: Feature validation
print(f"\nSTEP 8: Feature validation...")

# Check for any infinite values
infinite_cols = []
for col in df.columns:
    if df[col].dtype in ['float64', 'int64']:
        if np.isinf(df[col]).any():
            infinite_cols.append(col)

if infinite_cols:
    print(f"  ⚠️  WARNING: Infinite values found in: {infinite_cols}")
else:
    print(f"  ✅ No infinite values found")

# Check for any NaN values in new features
new_features = [col for col in df.columns if col not in ['age', 'sex', 'gender', 'hand', 'edu', 'country', 
                                                         'diagnosis_yes_no', 'diagnosis', 'autism_target',
                                                         'eq_total', 'sqr_total', 'aq_total']]
nan_features = []
for col in new_features:
    if df[col].isnull().any():
        nan_features.append(col)

if nan_features:
    print(f"  ⚠️  WARNING: NaN values found in new features: {nan_features}")
else:
    print(f"  ✅ No NaN values in new features")

# Show feature summary
print(f"\nFeature engineering summary:")
print(f"  Original features: {len(df.columns) - len(new_features)}")
print(f"  New features created: {len(new_features)}")
print(f"  Total features: {len(df.columns)}")

# Show sample of new features
print(f"\nSample of new features:")
sample_new_features = new_features[:10]  # Show first 10 new features
for col in sample_new_features:
    if col in df.columns:
        sample_vals = df[col].dropna().head(3).tolist()
        print(f"  {col}: {sample_vals}")

print(f"\nStep E complete. Dataset shape: {df.shape}")
print("✅ Age-based features created")
print("✅ Demographic features encoded")
print("✅ EQ-SQR interactions created (NO AQ interactions)")
print("✅ AQ threshold kept for sample exclusion criteria")
print("✅ Data ready for standardization")

### F. Data Standardization and Encoding (YBT Adapted)


In [ ]:
print("="*80)
print("STEP F: YBT DATA STANDARDIZATION AND ENCODING")
print("="*80)

# DEBUG: Check what columns we actually have
print(f"Starting dataset shape: {df.shape}")
print(f"Columns containing 'age': {[col for col in df.columns if 'age' in col.lower()]}")
print(f"First 10 columns: {list(df.columns)[:10]}")

# Check if age_group exists
if 'age_group' in df.columns:
    print(f"✅ age_group column found: {df['age_group'].value_counts().to_dict()}")
else:
    print(f"❌ age_group column NOT found")
    print(f"Available columns: {list(df.columns)}")

# STEP 1: Save AQ data before removing AQ features (CRITICAL FOR STEP H)
print(f"\nSTEP 1: Saving AQ data before removing AQ features...")
aq_cols = [col for col in df.columns if 'aq' in col.lower()]
if aq_cols:
    # Save AQ data with autism_target for Step H filtering
    aq_backup_cols = ['autism_target'] + aq_cols
    df_aq_backup = df[aq_backup_cols].copy()
    df_aq_backup.to_csv('data/processed/ybt_aq_backup.csv', index=True)
    print(f"  ✅ Saved AQ data backup: {df_aq_backup.shape}")
    print(f"  ✅ AQ columns saved: {aq_cols}")
else:
    print(f"  ⚠️  No AQ columns found to save")

# STEP 2: Remove data leakage columns (diagnosis columns)
print(f"\nSTEP 2: Removing data leakage columns...")
leakage_cols = ['diagnosis_yes_no', 'diagnosis', 'diagnosis_69_TEXT']
cols_to_remove = [col for col in leakage_cols if col in df.columns]
df = df.drop(columns=cols_to_remove)
print(f"  ✅ Removed {len(cols_to_remove)} data leakage columns: {cols_to_remove}")

# STEP 3: Drop unnecessary columns
print(f"\nSTEP 3: Dropping unnecessary columns...")
unnecessary_cols = ['Progress', 'Duration (in seconds)', 'Finished', 'RecordedDate', 'ResponseId']
cols_to_remove = [col for col in unnecessary_cols if col in df.columns]
df = df.drop(columns=cols_to_remove)
print(f"  ✅ Removed {len(cols_to_remove)} unnecessary columns: {cols_to_remove}")

# STEP 4: Create age_group if missing (FIX)
print(f"\nSTEP 4: Creating age_group if missing...")
if 'age_group' not in df.columns and 'age' in df.columns:
    df['age_group'] = pd.cut(df['age'], bins=[0, 25, 35, 45, 55, 100], 
                             labels=['18-25', '26-35', '36-45', '46-55', '56+'])
    print(f"  ✅ Created age_group column")
    print(f"  Age group distribution: {df['age_group'].value_counts().to_dict()}")
elif 'age_group' in df.columns:
    print(f"  ✅ age_group column already exists")
    print(f"  Age group distribution: {df['age_group'].value_counts().to_dict()}")
else:
    print(f"  ⚠️  No age column found to create age_group")

# STEP 5: Apply StandardScaler to questionnaire items (CRITICAL STEP)
print(f"\nSTEP 5: Standardizing questionnaire items...")
from sklearn.preprocessing import StandardScaler

# Find individual questionnaire items (not totals)
questionnaire_cols = [col for col in df.columns if col.startswith(('eq10_', 'sq10_', 'aq_'))]
individual_item_cols = [col for col in questionnaire_cols if not col.endswith('_total')]

print(f"  Found {len(individual_item_cols)} individual questionnaire items to standardize")
print(f"  Items: {individual_item_cols[:5]}...")  # Show first 5

# Apply StandardScaler to individual items
scaler = StandardScaler()
if individual_item_cols:
    df[individual_item_cols] = scaler.fit_transform(df[individual_item_cols])
    print(f"  ✅ Standardized {len(individual_item_cols)} individual questionnaire items")
    print(f"  ✅ Preserved raw totals (eq_total, sqr_total, aq_total) for filtering")
else:
    print(f"  ⚠️  No individual questionnaire items found to standardize")

# STEP 6: One-hot encode age groups
print(f"\nSTEP 6: One-hot encoding age groups...")
if 'age_group' in df.columns:
    df = pd.get_dummies(df, columns=['age_group'], drop_first=True)
    print(f"  ✅ Age groups one-hot encoded")
else:
    print(f"  ⚠️  No age_group column found")

# STEP 7: Convert remaining categorical columns to numeric
print(f"\nSTEP 7: Converting remaining categorical columns to numeric...")
categorical_cols = df.select_dtypes(include=['object']).columns
print(f"  Found categorical columns: {list(categorical_cols)}")

for col in categorical_cols:
    if col in df.columns:
        df[col] = pd.Categorical(df[col]).codes
        print(f"  ✅ {col} converted to numeric")

# STEP 8: Fill remaining NaNs
print(f"\nSTEP 8: Filling remaining NaNs...")
nan_count = df.isnull().sum().sum()
if nan_count > 0:
    df = df.fillna(0)
    print(f"  ✅ Filled {nan_count} NaN values with 0")
else:
    print(f"  ✅ No NaN values found")

# STEP 9: Final validation
print(f"\nSTEP 9: Final validation...")
print(f"  Dataset shape: {df.shape}")
print(f"  Data types: {df.dtypes.value_counts().to_dict()}")
print(f"  Missing values: {df.isnull().sum().sum()}")

# Check questionnaire items are now standardized
if individual_item_cols:
    sample_item = individual_item_cols[0]
    sample_vals = df[sample_item].head(5).tolist()
    print(f"  Sample standardized questionnaire item ({sample_item}): {sample_vals}")
    print(f"  ✅ Individual questionnaire items are now standardized")

# Check totals are preserved as raw
totals_cols = ['eq_total', 'sqr_total', 'aq_total']
available_totals = [col for col in totals_cols if col in df.columns]
if available_totals:
    sample_total = available_totals[0]
    sample_vals = df[sample_total].head(5).tolist()
    print(f"  Sample raw total ({sample_total}): {sample_vals}")
    print(f"  ✅ Raw totals preserved for filtering")

print(f"\nStep F complete. Dataset shape: {df.shape}")
print("✅ AQ data backup saved for Step H")
print("✅ Data leakage columns removed")
print("✅ Age groups created and one-hot encoded")
print("✅ Individual questionnaire items standardized")
print("✅ Raw totals preserved for filtering")
print("✅ Categorical columns converted to numeric")
print("✅ Data ready for balancing")

### G. Data Balancing (50/50 Split)


In [ ]:
print("="*80)
print("STEP G: YBT DATA BALANCING")
print("="*80)

# Check current dataset shape and target distribution
print(f"Starting dataset shape: {df.shape}")
target_distribution = df['autism_target'].value_counts()
print(f"Target distribution: {target_distribution.to_dict()}")
print(f"Autism prevalence: {target_distribution[1] / len(df) * 100:.2f}%")

# STEP 1: Analyze current class imbalance
print(f"\nSTEP 1: Analyzing class imbalance...")

autism_cases = df[df['autism_target'] == 1]
non_autism_cases = df[df['autism_target'] == 0]

print(f"  Autism cases: {len(autism_cases)}")
print(f"  Non-autism cases: {len(non_autism_cases)}")
print(f"  Imbalance ratio: {len(non_autism_cases) / len(autism_cases):.1f}:1")

# Check if balancing is needed
imbalance_ratio = len(non_autism_cases) / len(autism_cases)
if imbalance_ratio > 5:
    print(f"  ⚠️  HIGH IMBALANCE: {imbalance_ratio:.1f}:1 ratio")
    print(f"  Balancing recommended for better model performance")
elif imbalance_ratio > 2:
    print(f"  ⚠️  MODERATE IMBALANCE: {imbalance_ratio:.1f}:1 ratio")
    print(f"  Balancing may improve model performance")
else:
    print(f"  ✅ LOW IMBALANCE: {imbalance_ratio:.1f}:1 ratio")
    print(f"  Balancing may not be necessary")

# STEP 2: Apply data balancing (50/50 split)
print(f"\nSTEP 2: Applying data balancing...")

# Strategy: Undersample majority class to match minority class
# This creates a 50/50 split for better model training

print(f"  Strategy: Undersample majority class to create 50/50 split")
print(f"  Target: {len(autism_cases)} samples per class")

# Undersample non-autism cases to match autism cases
non_autism_undersampled = non_autism_cases.sample(n=len(autism_cases), random_state=42)
print(f"  Undersampled non-autism cases: {len(non_autism_undersampled)}")

# Combine balanced datasets
df_balanced = pd.concat([autism_cases, non_autism_undersampled], ignore_index=True)

# Shuffle the balanced dataset
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"  ✅ Balanced dataset created")
print(f"  Balanced dataset shape: {df_balanced.shape}")

# STEP 3: Save FULL dataset for Step H filtering (FIXED)
print(f"\nSTEP 3: Saving FULL dataset for Step H filtering...")

# Save the ENTIRE balanced dataset (not just AQ columns)
df_balanced.to_csv('data/processed/ybt_balanced_backup.csv', index=True)
print(f"  ✅ Saved FULL balanced dataset: {df_balanced.shape}")

# Also save AQ columns separately for reference
aq_cols = [col for col in df_balanced.columns if 'aq' in col.lower()]
if aq_cols:
    aq_backup_cols = ['autism_target'] + aq_cols
    df_aq_backup = df_balanced[aq_backup_cols].copy()
    df_aq_backup.to_csv('data/processed/ybt_aq_backup.csv', index=True)
    print(f"  ✅ Also saved AQ columns separately: {df_aq_backup.shape}")
    print(f"  ✅ AQ columns saved: {aq_cols}")
else:
    print(f"  ⚠️  No AQ columns found to save")

# STEP 4: Validate balanced dataset
print(f"\nSTEP 4: Validating balanced dataset...")

balanced_target_distribution = df_balanced['autism_target'].value_counts()
print(f"  Balanced target distribution: {balanced_target_distribution.to_dict()}")
print(f"  Balanced autism prevalence: {balanced_target_distribution[1] / len(df_balanced) * 100:.2f}%")

# Check if balancing was successful
if balanced_target_distribution[0] == balanced_target_distribution[1]:
    print(f"  ✅ Perfect 50/50 balance achieved")
else:
    print(f"  ⚠️  Balance not perfect: {balanced_target_distribution[0]}:{balanced_target_distribution[1]}")

# STEP 5: Check data quality after balancing
print(f"\nSTEP 5: Checking data quality after balancing...")

# Check for missing values
missing_values = df_balanced.isnull().sum().sum()
print(f"  Missing values: {missing_values}")

# Check data types
data_types = df_balanced.dtypes.value_counts()
print(f"  Data types: {data_types.to_dict()}")

# Check questionnaire totals are preserved
totals_cols = ['eq_total', 'sqr_total', 'aq_total']
available_totals = [col for col in totals_cols if col in df_balanced.columns]
if available_totals:
    print(f"  Questionnaire totals preserved:")
    for col in available_totals:
        sample_vals = df_balanced[col].head(3).tolist()
        print(f"    {col}: {sample_vals}")

# STEP 6: Show sample of balanced data
print(f"\nSTEP 6: Sample of balanced data...")

# Show sample from each class
autism_sample = df_balanced[df_balanced['autism_target'] == 1].head(3)
non_autism_sample = df_balanced[df_balanced['autism_target'] == 0].head(3)

print(f"  Sample autism cases:")
sample_cols = ['age', 'sex', 'eq_total', 'aq_total', 'autism_target']
for col in sample_cols:
    if col in df_balanced.columns:
        sample_vals = autism_sample[col].tolist()
        print(f"    {col}: {sample_vals}")

print(f"  Sample non-autism cases:")
for col in sample_cols:
    if col in df_balanced.columns:
        sample_vals = non_autism_sample[col].tolist()
        print(f"    {col}: {sample_vals}")

# STEP 7: Update main dataset
print(f"\nSTEP 7: Updating main dataset...")

# Replace original dataset with balanced version
df = df_balanced.copy()
print(f"  ✅ Main dataset updated with balanced data")
print(f"  Final dataset shape: {df.shape}")

# Final validation
print(f"\nFinal validation:")
print(f"  Dataset shape: {df.shape}")
print(f"  Target distribution: {df['autism_target'].value_counts().to_dict()}")
print(f"  Autism prevalence: {df['autism_target'].mean() * 100:.2f}%")
print(f"  Missing values: {df.isnull().sum().sum()}")

print(f"\nStep G complete. Dataset shape: {df.shape}")
print("✅ Data balanced to 50/50 split")
print("✅ FULL dataset backup saved for Step H")
print("✅ Dataset shuffled and ready for modeling")
print("✅ Data ready for final filtering")

### H. Final Dataset Filtering (Exclude Autism Cases with AQ < 6)


In [ ]:
print("="*80)
print("STEP H: YBT FINAL DATASET FILTERING (CORRECTED)")
print("="*80)

# Check current dataset shape and target distribution
print(f"Starting dataset shape: {df.shape}")
target_distribution = df['autism_target'].value_counts()
print(f"Target distribution: {target_distribution.to_dict()}")
print(f"Autism prevalence: {target_distribution[1] / len(df) * 100:.2f}%")

# STEP 1: Load FULL dataset from backup
print(f"\nSTEP 1: Loading FULL dataset from backup...")

try:
    # Load FULL balanced dataset (saved in Step G)
    df_with_aq = pd.read_csv('data/processed/ybt_balanced_backup.csv', index_col=0)
    print(f"  ✅ Loaded FULL balanced dataset: {df_with_aq.shape}")
    
    # Check if AQ columns exist
    aq_cols = [col for col in df_with_aq.columns if 'aq' in col.lower()]
    print(f"  AQ columns found: {len(aq_cols)}")
    
    if 'aq_total' not in df_with_aq.columns:
        print("  ❌ ERROR: aq_total column not found!")
        print("  Available columns:", list(df_with_aq.columns))
        raise ValueError("AQ data not available for filtering")
        
except Exception as e:
    print(f"  ❌ ERROR loading FULL dataset: {e}")
    print("  Cannot proceed with AQ-based filtering")
    print("  Current dataset will be used as-is")
    print(f"\nStep H complete. Dataset shape: {df.shape}")
    exit()

# STEP 2: Apply AQ-based filtering
print(f"\nSTEP 2: Applying AQ-based filtering...")

# Get current indices to map back to AQ data
current_indices = df.index
print(f"  Current dataset indices: {len(current_indices)} samples")

# Map AQ data to current dataset
df_with_aq_filtered = df_with_aq.loc[current_indices].copy()
print(f"  Mapped AQ data shape: {df_with_aq_filtered.shape}")

# Check AQ distribution before filtering
aq_distribution = df_with_aq_filtered['aq_total'].describe()
print(f"  AQ distribution before filtering:")
print(f"    Mean: {aq_distribution['mean']:.2f}")
print(f"    Min: {aq_distribution['min']:.2f}")
print(f"    Max: {aq_distribution['max']:.2f}")

# Apply AQ filtering: Keep only autism cases with AQ ≥ 6
print(f"  Applying AQ filtering: Keep only autism cases with AQ ≥ 6")

# Separate autism and non-autism cases
autism_cases = df_with_aq_filtered[df_with_aq_filtered['autism_target'] == 1]
non_autism_cases = df_with_aq_filtered[df_with_aq_filtered['autism_target'] == 0]

print(f"  Before filtering:")
print(f"    Autism cases: {len(autism_cases)}")
print(f"    Non-autism cases: {len(non_autism_cases)}")

# Filter autism cases: keep only those with AQ ≥ 6
autism_high_aq = autism_cases[autism_cases['aq_total'] >= 6]
autism_low_aq = autism_cases[autism_cases['aq_total'] < 6]

print(f"  Autism cases with AQ ≥ 6: {len(autism_high_aq)}")
print(f"  Autism cases with AQ < 6: {len(autism_low_aq)} (will be excluded)")

# Combine filtered autism cases with all non-autism cases
df_filtered = pd.concat([autism_high_aq, non_autism_cases], ignore_index=True)

print(f"  After filtering:")
print(f"    Total samples: {len(df_filtered)}")
print(f"    Autism cases: {len(autism_high_aq)}")
print(f"    Non-autism cases: {len(non_autism_cases)}")

# STEP 3: Remove AQ features to prevent data leakage
print(f"\nSTEP 3: Removing AQ features to prevent data leakage...")

# Remove AQ columns
aq_columns_to_remove = [
    'aq_1', 'aq_2', 'aq_3', 'aq_4', 'aq_5', 'aq_6', 'aq_7', 'aq_8', 'aq_9', 'aq_10',
    'aq_total', 'high_aq'
]

# Check which AQ columns actually exist in the dataset
existing_aq_cols = [col for col in aq_columns_to_remove if col in df_filtered.columns]
df_final = df_filtered.drop(columns=existing_aq_cols)
print(f"  ✅ Removed {len(existing_aq_cols)} AQ columns")
print(f"  Final dataset shape: {df_final.shape}")

# STEP 4: Rebalance dataset after filtering (50/50 split)
print(f"\nSTEP 4: Rebalancing dataset after filtering...")

# Check new target distribution
new_target_distribution = df_final['autism_target'].value_counts()
print(f"  New target distribution: {new_target_distribution.to_dict()}")

# Calculate imbalance
autism_count = new_target_distribution.get(1, 0)
non_autism_count = new_target_distribution.get(0, 0)
imbalance_ratio = non_autism_count / autism_count if autism_count > 0 else float('inf')

print(f"  Imbalance ratio: {imbalance_ratio:.1f}:1")

# Apply balancing to create 50/50 split
if autism_count > 0 and non_autism_count > 0:
    print(f"  ⚠️  Imbalance detected, applying 50/50 balancing...")
    
    # Undersample majority class to match minority class
    autism_cases_final = df_final[df_final['autism_target'] == 1]
    non_autism_cases_final = df_final[df_final['autism_target'] == 0]
    
    # Undersample non-autism cases to match autism cases
    non_autism_undersampled = non_autism_cases_final.sample(n=len(autism_cases_final), random_state=42)
    
    # Combine and shuffle
    df_final = pd.concat([autism_cases_final, non_autism_undersampled], ignore_index=True)
    df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)
    
    print(f"  ✅ Dataset rebalanced to 50/50")
    print(f"  Final target distribution: {df_final['autism_target'].value_counts().to_dict()}")
else:
    print(f"  ⚠️  Cannot balance: Missing autism or non-autism cases")

# STEP 5: Final validation
print(f"\nSTEP 5: Final validation...")

# Check for AQ features
remaining_aq_features = [col for col in df_final.columns if 'aq' in col.lower()]
if len(remaining_aq_features) > 0:
    print(f"  ❌ WARNING: {len(remaining_aq_features)} AQ features still present: {remaining_aq_features}")
else:
    print(f"  ✅ No AQ features present - data leakage prevented")

# Check data quality
missing_values = df_final.isnull().sum().sum()
print(f"  Missing values: {missing_values}")

# Check target distribution
final_target_distribution = df_final['autism_target'].value_counts()
print(f"  Final target distribution: {final_target_distribution.to_dict()}")
print(f"  Final autism prevalence: {final_target_distribution[1] / len(df_final) * 100:.2f}%")

# STEP 6: Update main dataset
print(f"\nSTEP 6: Updating main dataset...")

# Replace original dataset with filtered version
df = df_final.copy()
print(f"  ✅ Main dataset updated with filtered data")
print(f"  Final dataset shape: {df.shape}")

# Final summary
print(f"\n" + "="*80)
print("STEP H SUMMARY")
print("="*80)
print(f"✅ AQ-based filtering applied: Only autism cases with AQ ≥ 6 kept")
print(f"✅ AQ features removed: Data leakage prevented")
print(f"✅ Dataset rebalanced to 50/50: {df['autism_target'].value_counts().to_dict()}")
print(f"✅ Final dataset shape: {df.shape}")
print(f"✅ Ready for experiments")

print(f"\nStep H complete. Dataset shape: {df.shape}")

### I. Data Cleaning (Duplicate Removal, Zero Variance Features)


In [ ]:
print("="*80)
print("STEP I: YBT DATA CLEANING")
print("="*80)

# Check current dataset shape and target distribution
print(f"Starting dataset shape: {df.shape}")
target_distribution = df['autism_target'].value_counts()
print(f"Target distribution: {target_distribution.to_dict()}")
print(f"Autism prevalence: {target_distribution[1] / len(df) * 100:.2f}%")

# STEP 1: Remove duplicate rows
print(f"\nSTEP 1: Removing duplicate rows...")

# Check for duplicates
duplicates_before = df.duplicated().sum()
print(f"  Duplicates found: {duplicates_before}")

if duplicates_before > 0:
    df = df.drop_duplicates()
    print(f"  ✅ Removed {duplicates_before} duplicate rows")
    print(f"  Dataset shape after duplicate removal: {df.shape}")
else:
    print(f"  ✅ No duplicates found")

# STEP 2: Remove zero variance features
print(f"\nSTEP 2: Removing zero variance features...")

# Check for zero variance features
zero_variance_features = []
for col in df.columns:
    if col != 'autism_target':  # Skip target variable
        try:
            if df[col].var() == 0:
                zero_variance_features.append(col)
        except:
            # Handle non-numeric columns
            if df[col].nunique() <= 1:
                zero_variance_features.append(col)

print(f"  Zero variance features found: {len(zero_variance_features)}")
if zero_variance_features:
    print(f"  Features to remove: {zero_variance_features}")
    df = df.drop(columns=zero_variance_features)
    print(f"  ✅ Removed {len(zero_variance_features)} zero variance features")
    print(f"  Dataset shape after zero variance removal: {df.shape}")
else:
    print(f"  ✅ No zero variance features found")

# STEP 3: Final data quality checks
print(f"\nSTEP 3: Final data quality checks...")

# Check for missing values
missing_values = df.isnull().sum().sum()
print(f"  Missing values: {missing_values}")

# Check data types
data_types = df.dtypes.value_counts()
print(f"  Data types: {data_types.to_dict()}")

# Check for infinite values
infinite_values = np.isinf(df.select_dtypes(include=[np.number])).sum().sum()
print(f"  Infinite values: {infinite_values}")

# Check target distribution
final_target_distribution = df['autism_target'].value_counts()
print(f"  Final target distribution: {final_target_distribution.to_dict()}")
print(f"  Final autism prevalence: {final_target_distribution[1] / len(df) * 100:.2f}%")

# STEP 4: Feature summary
print(f"\nSTEP 4: Feature summary...")

# Count different types of features
numeric_features = df.select_dtypes(include=[np.number]).columns
categorical_features = df.select_dtypes(include=['object']).columns
boolean_features = df.select_dtypes(include=['bool']).columns

print(f"  Numeric features: {len(numeric_features)}")
print(f"  Categorical features: {len(categorical_features)}")
print(f"  Boolean features: {len(boolean_features)}")
print(f"  Total features: {len(df.columns)}")

# Show sample of final features
print(f"  Sample features: {list(df.columns)[:10]}...")

# STEP 5: Final validation
print(f"\nSTEP 5: Final validation...")

# Check if dataset is ready for modeling
if len(df) < 100:
    print(f"  ⚠️  WARNING: Very small dataset ({len(df)} samples)")
elif len(df.columns) < 10:
    print(f"  ⚠️  WARNING: Very few features ({len(df.columns)} features)")
else:
    print(f"  ✅ Dataset size adequate for modeling")

# Check target balance
balance_ratio = final_target_distribution[0] / final_target_distribution[1]
if abs(balance_ratio - 1.0) > 0.1:
    print(f"  ⚠️  WARNING: Dataset not perfectly balanced ({balance_ratio:.2f}:1)")
else:
    print(f"  ✅ Dataset is well balanced")

# Final summary
print(f"\n" + "="*80)
print("STEP I SUMMARY")
print("="*80)
print(f"✅ Duplicates removed: {duplicates_before} rows")
print(f"✅ Zero variance features removed: {len(zero_variance_features)} features")
print(f"✅ Data quality validated")
print(f"✅ Final dataset shape: {df.shape}")
print(f"✅ Target distribution: {final_target_distribution.to_dict()}")
print(f"✅ Ready for experiments")

print(f"\nStep I complete. Dataset shape: {df.shape}")
print("✅ Data cleaning completed")
print("✅ Dataset ready for experimental setups")
print("✅ All preprocessing steps completed")

# J quick descriptive stats to check data pre modelling

In [ ]:
print("="*80)
print("DESCRIPTIVE & DEMOGRAPHIC STATISTICS")
print("="*80)

# Check current dataset shape and target distribution
print(f"Dataset Overview:")
print(f"  Total samples: {len(df)}")
print(f"  Total features: {len(df.columns)}")
print(f"  Target distribution: {df['autism_target'].value_counts().to_dict()}")
print(f"  Autism prevalence: {df['autism_target'].mean() * 100:.2f}%")

# STEP 1: Demographic Statistics
print(f"\n" + "="*60)
print("DEMOGRAPHIC STATISTICS")
print("="*60)

# Age statistics
if 'age' in df.columns:
    print(f"\nAge Distribution:")
    age_stats = df['age'].describe()
    print(f"  Mean age: {age_stats['mean']:.1f} years")
    print(f"  Median age: {age_stats['50%']:.1f} years")
    print(f"  Age range: {age_stats['min']:.1f} - {age_stats['max']:.1f} years")
    print(f"  Standard deviation: {age_stats['std']:.1f} years")
    
    # Age by autism status
    autism_age = df[df['autism_target'] == 1]['age']
    non_autism_age = df[df['autism_target'] == 0]['age']
    print(f"  Autism cases - Mean age: {autism_age.mean():.1f} years")
    print(f"  Non-autism cases - Mean age: {non_autism_age.mean():.1f} years")

# Sex distribution
if 'sex' in df.columns:
    print(f"\nSex Distribution:")
    sex_dist = df['sex'].value_counts()
    print(f"  Overall: {sex_dist.to_dict()}")
    
    # Sex by autism status
    autism_sex = df[df['autism_target'] == 1]['sex'].value_counts()
    non_autism_sex = df[df['autism_target'] == 0]['sex'].value_counts()
    print(f"  Autism cases: {autism_sex.to_dict()}")
    print(f"  Non-autism cases: {non_autism_sex.to_dict()}")

# Education distribution
if 'edu' in df.columns:
    print(f"\nEducation Distribution:")
    edu_dist = df['edu'].value_counts()
    print(f"  Overall: {edu_dist.to_dict()}")
    
    # Education by autism status
    autism_edu = df[df['autism_target'] == 1]['edu'].value_counts()
    non_autism_edu = df[df['autism_target'] == 0]['edu'].value_counts()
    print(f"  Autism cases: {autism_edu.to_dict()}")
    print(f"  Non-autism cases: {non_autism_edu.to_dict()}")

# Country distribution (top 10)
if 'country' in df.columns:
    print(f"\nCountry Distribution (Top 10):")
    country_dist = df['country'].value_counts().head(10)
    print(f"  Overall: {country_dist.to_dict()}")
    
    # Country by autism status
    autism_country = df[df['autism_target'] == 1]['country'].value_counts().head(5)
    non_autism_country = df[df['autism_target'] == 0]['country'].value_counts().head(5)
    print(f"  Autism cases (top 5): {autism_country.to_dict()}")
    print(f"  Non-autism cases (top 5): {non_autism_country.to_dict()}")

# STEP 2: Questionnaire Statistics
print(f"\n" + "="*60)
print("QUESTIONNAIRE STATISTICS")
print("="*60)

# EQ-10 statistics
if 'eq_total' in df.columns:
    print(f"\nEQ-10 (Empathy Quotient) Statistics:")
    eq_stats = df['eq_total'].describe()
    print(f"  Mean EQ: {eq_stats['mean']:.2f}")
    print(f"  Median EQ: {eq_stats['50%']:.2f}")
    print(f"  EQ range: {eq_stats['min']:.1f} - {eq_stats['max']:.1f}")
    print(f"  Standard deviation: {eq_stats['std']:.2f}")
    
    # EQ by autism status
    autism_eq = df[df['autism_target'] == 1]['eq_total']
    non_autism_eq = df[df['autism_target'] == 0]['eq_total']
    print(f"  Autism cases - Mean EQ: {autism_eq.mean():.2f}")
    print(f"  Non-autism cases - Mean EQ: {non_autism_eq.mean():.2f}")
    print(f"  Difference: {autism_eq.mean() - non_autism_eq.mean():.2f}")

# SQR-10 statistics
if 'sqr_total' in df.columns:
    print(f"\nSQR-10 (Systemizing Quotient) Statistics:")
    sqr_stats = df['sqr_total'].describe()
    print(f"  Mean SQR: {sqr_stats['mean']:.2f}")
    print(f"  Median SQR: {sqr_stats['50%']:.2f}")
    print(f"  SQR range: {sqr_stats['min']:.1f} - {sqr_stats['max']:.1f}")
    print(f"  Standard deviation: {sqr_stats['std']:.2f}")
    
    # SQR by autism status
    autism_sqr = df[df['autism_target'] == 1]['sqr_total']
    non_autism_sqr = df[df['autism_target'] == 0]['sqr_total']
    print(f"  Autism cases - Mean SQR: {autism_sqr.mean():.2f}")
    print(f"  Non-autism cases - Mean SQR: {non_autism_sqr.mean():.2f}")
    print(f"  Difference: {autism_sqr.mean() - non_autism_sqr.mean():.2f}")

# STEP 3: Clinical Validation
print(f"\n" + "="*60)
print("CLINICAL VALIDATION")
print("="*60)

# Check if we have AQ data for validation (from backup)
try:
    df_aq_backup = pd.read_csv('data/processed/ybt_aq_backup.csv', index_col=0)
    current_indices = df.index
    df_aq_validation = df_aq_backup.loc[current_indices].copy()
    
    print(f"\nAQ-10 Validation (from backup data):")
    aq_stats = df_aq_validation['aq_total'].describe()
    print(f"  Mean AQ: {aq_stats['mean']:.2f}")
    print(f"  Median AQ: {aq_stats['50%']:.2f}")
    print(f"  AQ range: {aq_stats['min']:.1f} - {aq_stats['max']:.1f}")
    
    # AQ by autism status
    autism_aq = df_aq_validation[df_aq_validation['autism_target'] == 1]['aq_total']
    non_autism_aq = df_aq_validation[df_aq_validation['autism_target'] == 0]['aq_total']
    print(f"  Autism cases - Mean AQ: {autism_aq.mean():.2f}")
    print(f"  Non-autism cases - Mean AQ: {non_autism_aq.mean():.2f}")
    print(f"  Difference: {autism_aq.mean() - non_autism_aq.mean():.2f}")
    
    # Clinical interpretation
    if autism_aq.mean() > non_autism_aq.mean():
        print(f"  ✅ CLINICALLY CORRECT: Autism cases have higher AQ scores")
    else:
        print(f"  ❌ CLINICALLY COUNTERINTUITIVE: Autism cases have lower AQ scores")
    
    # Check AQ filtering effectiveness
    autism_high_aq = len(autism_aq[autism_aq >= 6])
    autism_low_aq = len(autism_aq[autism_aq < 6])
    print(f"  Autism cases with AQ ≥ 6: {autism_high_aq}")
    print(f"  Autism cases with AQ < 6: {autism_low_aq} (excluded)")
    print(f"  AQ filtering effectiveness: {autism_high_aq/(autism_high_aq+autism_low_aq)*100:.1f}% kept")
    
except Exception as e:
    print(f"  ⚠️  Could not load AQ data for validation: {e}")

# STEP 4: Data Quality Assessment
print(f"\n" + "="*60)
print("DATA QUALITY ASSESSMENT")
print("="*60)

# Check for missing values
missing_values = df.isnull().sum().sum()
print(f"Missing values: {missing_values}")

# Check for infinite values
infinite_values = np.isinf(df.select_dtypes(include=[np.number])).sum().sum()
print(f"Infinite values: {infinite_values}")

# Check data types
data_types = df.dtypes.value_counts()
print(f"Data types: {data_types.to_dict()}")

# Check for potential outliers
numeric_cols = df.select_dtypes(include=[np.number]).columns
outlier_summary = {}
for col in numeric_cols:
    if col != 'autism_target':
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        outliers = len(df[(df[col] < lower_bound) | (df[col] > upper_bound)])
        outlier_summary[col] = outliers

high_outlier_cols = {k: v for k, v in outlier_summary.items() if v > 10}
if high_outlier_cols:
    print(f"Columns with >10 outliers: {high_outlier_cols}")
else:
    print("No columns with excessive outliers detected")

# STEP 5: Summary and Recommendations
print(f"\n" + "="*60)
print("SUMMARY & RECOMMENDATIONS")
print("="*60)

print(f"Dataset Status:")
print(f"  ✅ Size: {len(df)} samples, {len(df.columns)} features")
print(f"  ✅ Balance: Perfect 50/50 split")
print(f"  ✅ Quality: No missing values, no infinite values")
print(f"  ✅ AQ filtering: Applied (autism cases with AQ ≥ 6)")

print(f"\nReady for Experiments:")
print(f"  ✅ Baseline models (Experiment A)")
print(f"  ✅ PCA analysis (Experiment B)")
print(f"  ✅ Threshold optimization (Experiment C)")
print(f"  ✅ AQ-based experiments (Experiment D)")

print(f"\n" + "="*80)
print("DESCRIPTIVE ANALYSIS COMPLETE")
print("="*80)

## 2. YBT EXPERIMENTAL SETUPS

### A. Baseline Models: Predicting Autism Target (Without AQ Items)


In [ ]:
print("="*80)
print("EXPERIMENT A: YBT BASELINE MODELS - COMPLETE AQ EXCLUSION")
print("="*80)

print(f"Dataset shape: {df.shape}")

# Prepare features and target
print("\nPreparing features and target...")

# AQ features should already be removed in Step F.5
# Verify no AQ features remain
aq_features_remaining = [col for col in df.columns if 'aq' in col.lower()]
print(f"AQ features remaining: {aq_features_remaining}")

if len(aq_features_remaining) > 0:
    print("⚠️  ERROR: AQ features still present! Remove them first.")
    df = df.drop(columns=aq_features_remaining, errors='ignore')
    print(f"Removed {len(aq_features_remaining)} remaining AQ features")
else:
    print("✅ No AQ features present - data leakage prevented")

# Create feature matrix excluding AQ-related features
feature_cols = [col for col in df.columns if col != 'autism_target']
print(f"Feature columns ({len(feature_cols)}): {feature_cols}")

X = df[feature_cols]
y = df['autism_target']

print(f"Feature matrix shape: {X.shape}")
print(f"Target distribution: {y.value_counts().to_dict()}")

# Handle data types and missing values
print("\nHandling data types and missing values...")
print(f"Missing values before: {X.isnull().sum().sum()}")

# Convert categorical columns to numeric
categorical_cols = X.select_dtypes(include=['object']).columns
print(f"Categorical columns found: {list(categorical_cols)}")

for col in categorical_cols:
    X[col] = pd.Categorical(X[col]).codes
    print(f"  Converted {col} to numeric codes")

# Fill any remaining missing values
X = X.fillna(0)
print(f"Missing values after: {X.isnull().sum().sum()}")

# Check for data leakage
print("\nChecking for data leakage...")
feature_correlations = X.corrwith(y).abs().sort_values(ascending=False)
print("Top 10 feature correlations with target:")
for i, (feature, corr) in enumerate(feature_correlations.head(10).items()):
    print(f"  {feature}: {corr:.4f}")

# Flag high correlations
high_corr_features = feature_correlations[feature_correlations > 0.7]
if len(high_corr_features) > 0:
    print(f"\n⚠️  WARNING: {len(high_corr_features)} features with high correlation (>0.7):")
    for feature, corr in high_corr_features.items():
        print(f"  {feature}: {corr:.4f}")
else:
    print("\n✅ No high correlation features (correlation < 0.7)")

# Investigate EQ data leakage specifically
print("\n🔍 INVESTIGATING EQ DATA LEAKAGE:")
eq_features = [col for col in X.columns if col.startswith('eq')]
if eq_features:
    eq_correlations = X[eq_features].corrwith(y).abs().sort_values(ascending=False)
    print(f"EQ feature correlations with autism target:")
    for feature, corr in eq_correlations.items():
        print(f"  {feature}: {corr:.4f}")
    
    # Check if EQ correlations are suspiciously high
    high_eq_corr = eq_correlations[eq_correlations > 0.2]  # Lowered threshold to 0.2 for investigation
    if len(high_eq_corr) > 0:
        print(f"\n⚠️  INVESTIGATING: {len(high_eq_corr)} EQ features with correlation > 0.2:")
        for feature, corr in high_eq_corr.items():
            print(f"  {feature}: {corr:.4f}")
        
        print("\n🔍 DETAILED EQ LEAKAGE ANALYSIS:")
        print("Checking if EQ features contain autism-related information...")
        
        # Load original data to check EQ responses by autism status
        try:
            df_original = pd.read_csv('data/processed/ybt_processed.csv')
            if 'autism_target' in df_original.columns:
                # Check EQ responses for autism vs non-autism cases
                autism_cases = df_original[df_original['autism_target'] == 1]
                non_autism_cases = df_original[df_original['autism_target'] == 0]
                
                print(f"\nEQ Response Analysis (Original Dataset):")
                print(f"Autism cases: {len(autism_cases)}")
                print(f"Non-autism cases: {len(non_autism_cases)}")
                
                # Check EQ total scores
                if 'eq_total' in df_original.columns:
                    autism_eq_mean = autism_cases['eq_total'].mean()
                    non_autism_eq_mean = non_autism_cases['eq_total'].mean()
                    print(f"\nEQ Total Scores:")
                    print(f"  Autism cases mean EQ: {autism_eq_mean:.2f}")
                    print(f"  Non-autism cases mean EQ: {non_autism_eq_mean:.2f}")
                    print(f"  Difference: {abs(autism_eq_mean - non_autism_eq_mean):.2f}")
                    
                    # Check if difference is clinically meaningful
                    if abs(autism_eq_mean - non_autism_eq_mean) > 0.5:
                        print(f"  ⚠️  SIGNIFICANT DIFFERENCE: EQ scores differ by >0.5 points")
                        print(f"  This may indicate that EQ features contain autism-related information")
                        print(f"  Consider excluding EQ features from baseline models")
                    else:
                        print(f"  ✅ MINOR DIFFERENCE: EQ scores differ by <0.5 points")
                        print(f"  This suggests EQ features are not strongly related to autism")
                
                # Check individual EQ items
                eq_items = [col for col in df_original.columns if col.startswith('eq10_')]
                if eq_items:
                    print(f"\nIndividual EQ Item Analysis:")
                    for item in eq_items[:5]:  # Check first 5 items
                        autism_item_mean = autism_cases[item].mean()
                        non_autism_item_mean = non_autism_cases[item].mean()
                        diff = abs(autism_item_mean - non_autism_item_mean)
                        print(f"  {item}: Autism={autism_item_mean:.2f}, Non-autism={non_autism_item_mean:.2f}, Diff={diff:.2f}")
                        
                        if diff > 0.3:
                            print(f"    ⚠️  High difference in {item}")
                        else:
                            print(f"    ✅ Normal difference in {item}")
            
        except Exception as e:
            print(f"Could not load original data for EQ analysis: {e}")
        
        print("\n📋 EQ LEAKAGE ASSESSMENT:")
        print("Based on the analysis above:")
        print("1. If EQ correlations are >0.2 AND EQ scores differ significantly between groups")
        print("2. Then EQ features may contain autism-related information (data leakage)")
        print("3. Consider excluding EQ features from baseline models")
        print("4. Or investigate further to understand the relationship")
        
    else:
        print("\n✅ EQ correlations appear normal (< 0.2)")
        print("No evidence of data leakage in EQ features")
else:
    print("No EQ features found in dataset")

# Scale features
print("\nScaling features...")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)
print(f"Scaled features shape: {X_scaled.shape}")
print(f"Feature means: {X_scaled.mean().mean():.6f}")
print(f"Feature stds: {X_scaled.std().mean():.6f}")

# Split data
print("\nSplitting data...")
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"Training target distribution: {y_train.value_counts().to_dict()}")
print(f"Test target distribution: {y_test.value_counts().to_dict()}")

# Define models
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss'),
    'LightGBM': LGBMClassifier(random_state=42, verbose=-1)
}

# Train and evaluate models
print("\n" + "="*60)
print("TRAINING AND EVALUATING BASELINE MODELS (COMPLETE AQ EXCLUSION)")
print("="*60)

results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    # Train model
    model.fit(X_train, y_train)
    
    # Make predictions
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_pred_proba)
    
    # Store results
    results[name] = {
        'accuracy': accuracy,
        'f1': f1,
        'precision': precision,
        'recall': recall,
        'auc': auc
    }
    
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  F1-score: {f1:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall: {recall:.4f}")
    print(f"  AUC: {auc:.4f}")

# Summary of results
print("\n" + "="*60)
print("YBT BASELINE MODELS SUMMARY (COMPLETE AQ EXCLUSION)")
print("="*60)

results_df = pd.DataFrame(results).T
results_df = results_df.sort_values('auc', ascending=False)

print("\nResults ranked by AUC:")
print(results_df.round(4))

# Save results
results_df.to_csv('data/processed/ybt_baseline_models_complete_aq_exclusion_results.csv')
print(f"\nResults saved to: data/processed/ybt_baseline_models_complete_aq_exclusion_results.csv")

print("\n" + "="*80)
print("EXPERIMENT A COMPLETE (COMPLETE AQ EXCLUSION)")
print("="*80)

### B. PCA Analysis (YBT Adapted - No SPQ)


In [ ]:
print("="*80)
print("EXPERIMENT B: YBT PCA ANALYSIS (NO SPQ)")
print("="*80)

# Prepare features for PCA (EQ and SQR items only - no SPQ, no AQ)
print("Preparing features for PCA analysis...")

# Get questionnaire item columns (excluding totals and AQ items)
eq_items = [col for col in df.columns if col.startswith('eq10_')]
sqr_items = [col for col in df.columns if col.startswith('sq10_')]

print(f"EQ items: {len(eq_items)} - {eq_items}")
print(f"SQR items: {len(sqr_items)} - {sqr_items}")

# Combine EQ and SQR items for PCA
pca_features = eq_items + sqr_items
print(f"Total PCA features: {len(pca_features)}")

if len(pca_features) == 0:
    print("ERROR: No questionnaire items found for PCA!")
    print("Available columns:", list(df.columns))
else:
    # Prepare data for PCA
    X_pca = df[pca_features].copy()
    y_pca = df['autism_target']
    
    print(f"PCA dataset shape: {X_pca.shape}")
    print(f"Target distribution: {y_pca.value_counts().to_dict()}")
    
    # Handle missing values
    X_pca = X_pca.fillna(X_pca.median())
    
    # Scale features
    scaler_pca = StandardScaler()
    X_pca_scaled = scaler_pca.fit_transform(X_pca)
    
    # Apply PCA
    print("\nApplying PCA...")
    pca = PCA()
    X_pca_transformed = pca.fit_transform(X_pca_scaled)
    
    # Analyze explained variance
    explained_variance_ratio = pca.explained_variance_ratio_
    cumulative_variance = np.cumsum(explained_variance_ratio)
    
    print(f"Explained variance by component:")
    for i, (var, cum_var) in enumerate(zip(explained_variance_ratio, cumulative_variance)):
        print(f"  PC{i+1}: {var:.4f} (cumulative: {cum_var:.4f})")
    
    # Find optimal number of components (95% variance)
    n_components_95 = np.argmax(cumulative_variance >= 0.95) + 1
    print(f"\nComponents needed for 95% variance: {n_components_95}")
    
    # Apply PCA with optimal components
    pca_optimal = PCA(n_components=n_components_95)
    X_pca_optimal = pca_optimal.fit_transform(X_pca_scaled)
    
    print(f"PCA-reduced features shape: {X_pca_optimal.shape}")
    
    # Train models with PCA features
    print("\nTraining models with PCA features...")
    
    # Split data
    X_train_pca, X_test_pca, y_train_pca, y_test_pca = train_test_split(
        X_pca_optimal, y_pca, test_size=0.2, random_state=42, stratify=y_pca
    )
    
    # Define models
    models_pca = {
        'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
        'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100),
        'Gradient Boosting': GradientBoostingClassifier(random_state=42),
        'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss'),
        'LightGBM': LGBMClassifier(random_state=42, verbose=-1)
    }
    
    # Train and evaluate models
    results_pca = {}
    
    for name, model in models_pca.items():
        print(f"\nTraining {name} with PCA features...")
        
        # Train model
        model.fit(X_train_pca, y_train_pca)
        
        # Make predictions
        y_pred_pca = model.predict(X_test_pca)
        y_pred_proba_pca = model.predict_proba(X_test_pca)[:, 1]
        
        # Calculate metrics
        accuracy = accuracy_score(y_test_pca, y_pred_pca)
        f1 = f1_score(y_test_pca, y_pred_pca)
        precision = precision_score(y_test_pca, y_pred_pca)
        recall = recall_score(y_test_pca, y_pred_pca)
        auc = roc_auc_score(y_test_pca, y_pred_proba_pca)
        
        # Store results
        results_pca[name] = {
            'accuracy': accuracy,
            'f1': f1,
            'precision': precision,
            'recall': recall,
            'auc': auc
        }
        
        print(f"  Accuracy: {accuracy:.4f}")
        print(f"  F1-score: {f1:.4f}")
        print(f"  Precision: {precision:.4f}")
        print(f"  Recall: {recall:.4f}")
        print(f"  AUC: {auc:.4f}")
    
    # Summary of PCA results
    print("\n" + "="*60)
    print("YBT PCA MODELS SUMMARY")
    print("="*60)
    
    results_pca_df = pd.DataFrame(results_pca).T
    results_pca_df = results_pca_df.sort_values('auc', ascending=False)
    
    print("\nPCA Results ranked by AUC:")
    print(results_pca_df.round(4))
    
    # Save PCA results
    results_pca_df.to_csv('data/processed/ybt_pca_models_results.csv')
    print(f"\nPCA results saved to: data/processed/ybt_pca_models_results.csv")

print("\n" + "="*80)
print("EXPERIMENT B COMPLETE (PCA ANALYSIS)")
print("="*80)


## 3. DESCRIPTIVE ANALYSIS & DATA VALIDATION

Before proceeding to additional experiments, let's conduct comprehensive descriptive analysis to validate our final dataset.


In [ ]:
print("="*80)
print("COMPREHENSIVE DESCRIPTIVE ANALYSIS OF FINAL YBT DATASET")
print("="*80)

# 1. Dataset Overview
print("1. DATASET OVERVIEW")
print("-" * 40)
print(f"Final dataset shape: {df.shape}")
print(f"Total samples: {len(df)}")
print(f"Total features: {len(df.columns)}")
print(f"Target distribution: {df['autism_target'].value_counts().to_dict()}")
print(f"Autism percentage: {df['autism_target'].mean()*100:.2f}%")

# 2. Questionnaire Score Distributions
print("\n2. QUESTIONNAIRE SCORE DISTRIBUTIONS")
print("-" * 40)
questionnaire_totals = ['eq_total', 'sqr_total', 'aq_total', 'd_score']
for col in questionnaire_totals:
    if col in df.columns:
        print(f"\n{col.upper()}:")
        print(f"  Range: {df[col].min():.1f} - {df[col].max():.1f}")
        print(f"  Mean: {df[col].mean():.2f} ± {df[col].std():.2f}")
        print(f"  Median: {df[col].median():.2f}")
        
        # By autism status
        autism_scores = df[df['autism_target']==1][col]
        non_autism_scores = df[df['autism_target']==0][col]
        print(f"  Autism cases: {autism_scores.mean():.2f} ± {autism_scores.std():.2f}")
        print(f"  Non-autism cases: {non_autism_scores.mean():.2f} ± {non_autism_scores.std():.2f}")

# 3. Clinical Validation
print("\n3. CLINICAL VALIDATION")
print("-" * 40)

# Load AQ data temporarily for validation (from before data leakage prevention)
print("Loading AQ data for clinical validation...")
try:
    # Try to load the dataset before AQ removal
    df_with_aq = pd.read_csv('data/processed/ybt_processed.csv')
    if 'aq_total' in df_with_aq.columns:
        # Match the current dataset with the AQ data
        df_validation = df.copy()
        
        # Get the indices that correspond to our current filtered dataset
        # We need to map the current dataset back to the original AQ data
        current_indices = df.index
        df_validation['aq_total'] = df_with_aq.loc[current_indices, 'aq_total'].values
        
        # AQ clinical threshold analysis
        high_aq_cases = len(df_validation[df_validation['aq_total'] >= 6])
        autism_high_aq = len(df_validation[(df_validation['autism_target']==1) & (df_validation['aq_total'] >= 6)])
        autism_low_aq = len(df_validation[(df_validation['autism_target']==1) & (df_validation['aq_total'] < 6)])
        
        print(f"AQ Clinical Threshold (≥6):")
        print(f"  Total high AQ cases: {high_aq_cases} ({high_aq_cases/len(df_validation)*100:.1f}%)")
        print(f"  Autism cases with high AQ: {autism_high_aq}")
        print(f"  Autism cases with low AQ: {autism_low_aq}")
        if autism_high_aq + autism_low_aq > 0:
            print(f"  Autism high AQ rate: {autism_high_aq/(autism_high_aq+autism_low_aq)*100:.1f}%")
        
        # Clinical interpretation
        print(f"\nClinical Interpretation:")
        autism_mean_aq = df_validation[df_validation['autism_target']==1]['aq_total'].mean()
        non_autism_mean_aq = df_validation[df_validation['autism_target']==0]['aq_total'].mean()
        print(f"  Autism cases mean AQ: {autism_mean_aq:.2f}")
        print(f"  Non-autism cases mean AQ: {non_autism_mean_aq:.2f}")
        print(f"  Clinical expectation: Autism cases should have HIGHER AQ scores")
        print(f"  Current finding: {'✅ CORRECT' if autism_mean_aq > non_autism_mean_aq else '❌ COUNTERINTUITIVE'}")
        
        # Validation of filtering
        print(f"\nFiltering Validation:")
        print(f"  Expected: All autism cases should have AQ ≥ 6 (after filtering)")
        print(f"  Actual: {autism_low_aq} autism cases with AQ < 6")
        print(f"  Status: {'✅ FILTERING WORKED' if autism_low_aq == 0 else '❌ FILTERING FAILED'}")
        
        # Additional validation checks
        print(f"\nAdditional Validation Checks:")
        print(f"  AQ score range: {df_validation['aq_total'].min()}-{df_validation['aq_total'].max()} (should be 0-10)")
        print(f"  Total samples: {len(df_validation)}")
        print(f"  Autism samples: {len(df_validation[df_validation['autism_target']==1])}")
        print(f"  Non-autism samples: {len(df_validation[df_validation['autism_target']==0])}")
        
        # Check if the dataset is balanced
        target_balance = df_validation['autism_target'].value_counts()
        if len(target_balance) == 2:
            balance_ratio = target_balance[1] / target_balance[0]
            print(f"  Target balance ratio: {balance_ratio:.2f} (should be close to 1.0 for balanced dataset)")
            print(f"  Balance status: {'✅ BALANCED' if 0.8 <= balance_ratio <= 1.2 else '❌ IMBALANCED'}")
        
    else:
        print("❌ AQ data not available for validation")
        print("Cannot perform clinical validation without AQ scores")
        
except Exception as e:
    print(f"❌ Could not load AQ data for validation: {e}")
    print("This indicates a problem with the data processing pipeline")
    print("Check that AQ scoring was completed and data was saved correctly")

# 4. Feature Correlation Analysis
print("\n4. FEATURE CORRELATION ANALYSIS")
print("-" * 40)
# Get numeric features only
numeric_features = df.select_dtypes(include=[np.number]).columns
numeric_features = [col for col in numeric_features if col != 'autism_target']

# Calculate correlations with target
correlations = df[numeric_features].corrwith(df['autism_target']).abs().sort_values(ascending=False)
print("Top 10 features correlated with autism target:")
for i, (feature, corr) in enumerate(correlations.head(10).items()):
    print(f"  {i+1:2d}. {feature}: {corr:.4f}")

# 5. Data Quality Checks
print("\n5. DATA QUALITY CHECKS")
print("-" * 40)
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"Duplicate rows: {df.duplicated().sum()}")
print(f"Infinite values: {np.isinf(df.select_dtypes(include=[np.number])).sum().sum()}")

# Check for constant features
constant_features = []
for col in numeric_features:
    if df[col].nunique() <= 1:
        constant_features.append(col)
print(f"Constant features: {len(constant_features)} - {constant_features}")

print("\n" + "="*80)
print("DESCRIPTIVE ANALYSIS COMPLETE")
print("="*80)


# validation cell

In [ ]:
print("="*80)
print("FINAL VALIDATION: DATA LEAKAGE CHECK")
print("="*80)

# Check for any remaining AQ features
aq_features_check = [col for col in df.columns if 'aq' in col.lower()]
print(f"AQ features in final dataset: {aq_features_check}")

if len(aq_features_check) > 0:
    print("❌ DATA LEAKAGE DETECTED!")
    print("The following AQ features are still present:")
    for feature in aq_features_check:
        print(f"  - {feature}")
    print("\nThis will invalidate the model results!")
else:
    print("✅ No AQ features present - data leakage prevented")

# Check feature correlations with target
print("\nFeature correlations with autism target:")
numeric_features = df.select_dtypes(include=[np.number]).columns
numeric_features = [col for col in numeric_features if col != 'autism_target']

correlations = df[numeric_features].corrwith(df['autism_target']).abs().sort_values(ascending=False)
print("Top 5 feature correlations:")
for i, (feature, corr) in enumerate(correlations.head(5).items()):
    print(f"  {i+1}. {feature}: {corr:.4f}")

# Flag any suspiciously high correlations
high_corr_features = correlations[correlations > 0.5]
if len(high_corr_features) > 0:
    print(f"\n⚠️  WARNING: {len(high_corr_features)} features with very high correlation (>0.5):")
    for feature, corr in high_corr_features.items():
        print(f"  {feature}: {corr:.4f}")
else:
    print("\n✅ No suspiciously high correlations detected")

print("\n" + "="*80)
print("VALIDATION COMPLETE")
print("="*80)